In [0]:
%run ./00_configuration

## Document contract and utilities (domain-agnostic)

Pure Python, no Spark writes in this section. `parse_document` validates any source file against the metadata contract configured in `00_configuration` (`CORE_METADATA_FIELDS` + `DOMAIN_ATTRIBUTES`), so retargeting the framework to another domain requires a configuration change only — the contract code stays the same. Utilities cover hashing, section-aware chunking (parameters from configuration), registry validation, the keyword fallback, extractive responses, and embedding parsing/cosine helpers.

In [0]:
"""Pure Python document contract, chunking, validation, and fallback utilities."""
import hashlib


def digest(text):
    """SHA-256 hex digest of a UTF-8 string.

    Defines content identity for versions, chunk IDs, and quarantine
    deduplication: identical text always maps to the same digest.
    """
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def parse_document(text):
    """Parse and validate one file against the embedded-metadata contract.

    Raises ValueError with the first failing rule (bad header, missing or
    non-string metadata field, non-ISO date, empty body) so ingestion can
    quarantine the file with a readable reason. Returns metadata, body, and
    content_hash of the FULL file text (metadata and body): any edit anywhere
    yields a different hash, so a file can never silently replace a version.
    """
    match = re.match(r'\A---metadata\s*\n(.*?)\n---\s*\n(.*)\Z', text, re.S)
    if not match:
        raise ValueError('Expected ---metadata, JSON block, ---, then document body')
    meta, body = json.loads(match[1]), match[2].strip()
    required = METADATA_FIELDS  # document identity + configured domain attributes
    for key in required:
        if not isinstance(meta.get(key), str) or not meta[key].strip():
            raise ValueError(f'Missing or invalid {key}')
    date.fromisoformat(meta['document_date'])
    if not body:
        raise ValueError('Empty document body')
    return {**{k: meta[k].strip() for k in required}, 'body': body, 'content_hash': digest(text)}


def chunk_document(record, max_words=None, overlap=None):
    """Split one accepted version into section-aware chunks.

    Splits on top-level '#' headings; long sections are windowed to at most
    max_words body words (CHUNK_MAX_WORDS by default) with CHUNK_OVERLAP_WORDS
    overlap; the record's title is prepended to every chunk's searchable text.
    chunk_id digests business key + version + content_hash + section and window
    position, so identical content always yields the same chunk row (idempotent
    MERGEs) while any content change produces new chunk IDs. 'title' belongs to
    the default DOMAIN_ATTRIBUTES; a deployment that removes it must adapt this
    prepend step.
    """
    max_words = CHUNK_MAX_WORDS if max_words is None else max_words
    overlap = CHUNK_OVERLAP_WORDS if overlap is None else overlap
    if not 0 <= overlap < max_words:
        raise ValueError('Invalid overlap')
    chunks = []
    for section_number, section in enumerate(re.split(r'(?m)(?=^# )', record['body'])):
        words = section.split()
        if not words:
            continue
        for start in range(0, len(words), max_words-overlap):
            text = record['title'] + '\n' + ' '.join(words[start:start+max_words])
            chunks.append(dict(
                **{k:v for k,v in record.items() if k != 'body'},
                chunk_id=digest(record['document_id']+'|'+record['version_id']+'|'+
                                record['content_hash']+'|'+str(section_number)+'|'+str(start)),
                section_heading=section.strip().splitlines()[0].lstrip('# '),
                chunk_text=text))
            if start + max_words >= len(words):
                break
    return chunks


def validate_candidates(candidates, registry):
    """Never pass stale text to a model. registry maps business ID to version/hash."""
    return [c for c in candidates
            if registry.get(c['document_id']) == (c['version_id'], c['content_hash'])]


def rank_chunks(question, chunks, limit=5):
    """Standard-library TF-IDF cosine retrieval; no model or vector service."""
    import math
    from collections import Counter
    stop = {'a','an','the','is','are','what','which','for','of','to','in','and','please','tell','me'}
    def terms(s):
        return [w for w in re.findall(r'[a-z0-9]+', s.lower()) if w not in stop]
    query = Counter(terms(question))
    if not query or not chunks:
        return []
    docs = [Counter(terms(c['chunk_text'])) for c in chunks]
    df = Counter(w for doc in docs for w in doc)
    n = len(docs)
    def vector(counts):
        return {w:(1+math.log(count))* (math.log((1+n)/(1+df[w]))+1)
                for w,count in counts.items()}
    q = vector(query)
    qnorm = math.sqrt(sum(v*v for v in q.values()))
    scored = []
    for chunk, counts in zip(chunks,docs):
        v = vector(counts)
        norm = math.sqrt(sum(x*x for x in v.values()))
        score = sum(q.get(w,0)*x for w,x in v.items())/(qnorm*norm) if norm else 0
        if score > 0:
            scored.append(dict(chunk, retrieval_score=score))
    return sorted(scored,key=lambda c:(-c['retrieval_score'],c['chunk_id']))[:limit]


def extractive_response(question, evidence):
    """Return source excerpts, not generated facts or an LLM answer."""
    if not evidence:
        return {'answer':'No matching current document excerpt was found.', 'sources':[], 'mode':'EXTRACTIVE'}
    sources, excerpts = [], []
    for i,e in enumerate(evidence,1):
        source = {key:e[key] for key in ('document_id','version_id','document_date','filename','source_path','section_heading')}
        sources.append(source)
        excerpts.append(f"[{i}] {e['chunk_text']}")
    return {'answer':'Current document excerpts:\n\n'+'\n\n'.join(excerpts),
            'sources':sources,'mode':'EXTRACTIVE'}


def parse_embedding(value):
    """Normalize common SQL embedding results to a finite, nonzero float vector."""
    import math
    if isinstance(value,str): value=json.loads(value)
    if isinstance(value,dict):
        if 'embedding' in value: value=value['embedding']
        elif 'data' in value and value['data']: value=value['data'][0]['embedding']
    if not isinstance(value,(list,tuple)) or not value:
        raise ValueError('Embedding response is not a nonempty vector')
    vector=[float(x) for x in value]
    if not all(math.isfinite(x) for x in vector) or not any(vector):
        raise ValueError('Embedding contains nonfinite values or is zero')
    return vector


def cosine_similarity(a,b):
    import math
    if len(a)!=len(b): raise ValueError('Embedding dimension mismatch')
    denom=math.sqrt(sum(x*x for x in a)*sum(x*x for x in b))
    return sum(x*y for x,y in zip(a,b))/denom if denom else 0.0

## Storage layer: Delta tables, manifest, and registry views

`ensure_storage()` creates six Delta tables (created only when absent, so reruns are safe) and two views:

* `document_versions` — immutable accepted version history (metadata, body, hash, source file)
* `document_chunks` — chunk history for **every** accepted version, active or not
* `active_document_chunks` — the retrieval source, refreshed to hold active-version chunks only (Change Data Feed enabled so a Delta Sync index could consume it later)
* `chunk_embeddings` — persisted embeddings keyed by `(chunk_id, model)`
* `quarantined_documents` — rejected source files with a reason and observation timestamp
* `retrieval_audit` — one row per retrieval request (question, mode, candidate and accepted refs, rejected count)
* `document_manifest` (view) — version history with rank and computed `is_active`
* `active_document_registry` (view) — current document ID, version, hash, and date

Table schemas are generated from the metadata contract in `00_configuration`, so adding a domain attribute grows every history table consistently.

In [0]:
def ensure_storage():
    """Create the project's Delta tables and views when absent (idempotent).

    Tables are created only when absent, so reruns never duplicate or overwrite
    history; the manifest and registry views are replaced so their definition
    always matches the current code.
    """
    for name, schema in [
        ("document_versions", VERSION_SCHEMA),
        ("document_chunks", CHUNK_SCHEMA),
        ("active_document_chunks", CHUNK_SCHEMA),
        ("chunk_embeddings", "chunk_id string, model string, embedding_json string"),
        ("quarantined_documents", QUARANTINE_SCHEMA),
        ("retrieval_audit", AUDIT_SCHEMA)]:
        if not spark.catalog.tableExists(table(name)):
            spark.createDataFrame([], schema).write.format("delta").saveAsTable(table(name))
    spark.sql(f"ALTER TABLE {table('active_document_chunks')} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    # Manifest is a view over immutable, accepted versions: activation is always
    # recomputed from history, never stored as a flag in a file. Equal-date ambiguity
    # is rejected during ingest BEFORE rows reach this view. Ordering: versions with an
    # eligible date first (future versions cannot win), then newest business date, then
    # version_id as a deterministic tiebreak. Arrival time and filename never participate.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('document_manifest')} AS
     SELECT *, version_rank = 1 AND document_date <= CAST(current_date() AS STRING) AS is_active
     FROM (
       SELECT *, ROW_NUMBER() OVER (PARTITION BY document_id
         ORDER BY CASE WHEN document_date <= CAST(current_date() AS STRING) THEN 0 ELSE 1 END,
                  document_date DESC, version_id) AS version_rank
       FROM {table('document_versions')}
     )""")
    # Single source of truth: retrieval must match every candidate against this
    # registry's (version_id, content_hash) before any evidence reaches a model.
    spark.sql(f"""CREATE OR REPLACE VIEW {table('active_document_registry')} AS
     SELECT document_id, version_id, content_hash, document_date
     FROM {table('document_manifest')} WHERE is_active""")

ensure_storage()
print("Storage ensured:", f"{CATALOG}.{SCHEMA}")